# Analysis of ODP geospatial names
**Author**:  Sian Teesdale <br>
**Date**:  5th August 2026 <br>
**Dataset Scope**: `ODP` <br>
**Report Type**: Ad-hoc analysis <br>

## Purpose

Explore the `name` field across ODP geography datasets (article-4-direction-area, conservation-area, listed-building-outline, tree-preservation-zone, tree) to characterise how "weird" the data is - exact duplicate names, blank/missing names, names that are bare reference codes rather than descriptions, and repeated boilerplate/placeholder phrases (e.g. "Town and Country...", "No name for this Entry").

This is a first pass exploration of how widespread the problems are, before deciding on whether we should raise this as an issue to LPAs through Check and Provide.


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import re
import urllib
from collections import Counter
from datetime import datetime

td = datetime.today().strftime('%Y-%m-%d')

pd.set_option("display.max_rows", 100)

data_dir = "../../data/db_downloads/"
os.makedirs(data_dir, exist_ok=True)


## Data Import

In [3]:
DATASET_SLUGS = [
    "article-4-direction-area",
    "conservation-area",
    "listed-building-outline",
    "tree-preservation-zone",
    "tree",
]

dfs = {
    slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
    for slug in DATASET_SLUGS
}

{slug: df.shape for slug, df in dfs.items()}


/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_6716/2165998576.py:10: DtypeWarning: Columns (14,15,19) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_6716/2165998576.py:10: DtypeWarning: Columns (15,21) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_6716/2165998576.py:10: DtypeWarning: Columns (14,16,17,22,24) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")


{'article-4-direction-area': (7326, 21),
 'conservation-area': (11013, 20),
 'listed-building-outline': (126247, 22),
 'tree-preservation-zone': (102574, 22),
 'tree': (262304, 25)}

## Analysis

### `name` column patterns

Reusable checks applied across all datasets: exact-duplicate names, common repeated phrases (e.g. boilerplate text like "Town and Country"), and names that look like bare reference codes rather than descriptions.

In [4]:
def word_ngrams(text, n):
    words = re.findall(r"[a-z0-9\']+", text.lower())
    return {' '.join(words[i:i + n]) for i in range(len(words) - n + 1)}


def duplicate_names(df, name_col='name'):
    """Names reused across more than one row."""
    counts = df[name_col].value_counts()
    return counts[counts > 1]


def top_ngrams(df, name_col='name', n=2, min_count=5, top=15):
    """Most common n-word phrases, counted by number of distinct rows containing them."""
    names = df[name_col].dropna().astype(str)
    counter = Counter()
    for text in names:
        counter.update(word_ngrams(text, n))
    return [(phrase, count) for phrase, count in counter.most_common(top) if count >= min_count]


def code_like_names(df, name_col='name'):
    """Names that look like a bare reference code (e.g. '0164B2', '11/222', '16.018') rather than a description.
    Requires at least one digit so plain single-word place names (e.g. 'Napsbury') aren't misflagged."""
    return df[df[name_col].str.match(r'^(?=.*[0-9])[A-Za-z0-9./-]{1,20}$', na=False)]


def missing_names(df, name_col='name'):
    """Rows with a blank/missing name - covers both NaN cells and empty or whitespace-only strings."""
    return df[df[name_col].isna() | (df[name_col].astype(str).str.strip() == '')]


### Per-dataset summary

In [84]:
for slug, df in dfs.items():
    n = len(df)
    dupes = duplicate_names(df)
    codes = code_like_names(df)
    blanks = missing_names(df)
    print(f"=== {slug} ({n} rows) ===")
    print(f"  {len(dupes)} distinct names reused across {dupes.sum()} rows ({dupes.sum() / n:.1%})")
    print(f"  {len(codes)} rows ({len(codes) / n:.1%}) look like bare reference codes")
    print(f"  {len(blanks)} rows ({len(blanks) / n:.1%}) have a blank/missing name")
    for gram_n in (2, 3):
        phrases = top_ngrams(df, n=gram_n)
        if phrases:
            top_phrase, top_count = phrases[0]
            print(f"  top {gram_n}-word phrase: '{top_phrase}' in {top_count} rows ({top_count / n:.1%})")
    print()


=== article-4-direction-area (7276 rows) ===
  460 distinct names reused across 2335 rows (32.1%)
  104 rows (1.4%) look like bare reference codes
  6 rows (0.1%) have a blank/missing name
  top 2-word phrase: 'article 4' in 778 rows (10.7%)
  top 3-word phrase: 'centre related a4d' in 582 rows (8.0%)

=== conservation-area (11008 rows) ===
  781 distinct names reused across 1807 rows (16.4%)
  10 rows (0.1%) look like bare reference codes
  110 rows (1.0%) have a blank/missing name
  top 2-word phrase: 'conservation area' in 1477 rows (13.4%)
  top 3-word phrase: 'road conservation area' in 40 rows (0.4%)

=== listed-building-outline (126199 rows) ===
  7022 distinct names reused across 25082 rows (19.9%)
  144 rows (0.1%) look like bare reference codes
  2245 rows (1.8%) have a blank/missing name
  top 2-word phrase: 'church of' in 5879 rows (4.7%)
  top 3-word phrase: 'church of st' in 4764 rows (3.8%)

=== tree-preservation-zone (102170 rows) ===
  7500 distinct names reused across

---

### Drill down into a dataset

Swap `dataset` below to inspect a specific dataset's full n-gram breakdown, then a chosen `phrase` to see the matching rows.

#### Article 4 direction area

In [110]:
dataset = "article-4-direction-area" # article-4-direction-area, conservation-area, listed-building-outlines, tree-preservation-zone, trees

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'article-4-direction-area' ---
   1511  (20.8%)  road
   1225  (16.9%)  land
   1015  (14.0%)  liverpool
    949  (13.1%)  street
    882  (12.1%)  4
    800  (11.0%)  direction
    778  (10.7%)  article
    774  (10.6%)  of
    722  (9.9%)  centre
    679  (9.3%)  and
    650  (8.9%)  a4d
    649  (8.9%)  at
    622  (8.6%)  related
    588  (8.1%)  lane
    483  (6.6%)  the

--- Most common 2-word phrases in 'article-4-direction-area' ---
    778  (10.7%)  article 4
    622  (8.6%)  related a4d
    604  (8.3%)  land at
    582  (8.0%)  centre related
    475  (6.5%)  4 direction
    379  (5.2%)  district centre
    316  (4.3%)  road liverpool
    256  (3.5%)  street liverpool
    214  (2.9%)  modified direction
    180  (2.5%)  site allocation
    142  (2.0%)  conservation area
    136  (1.9%)  major centre
    135  (1.9%)  town centres
    134  (1.8%)  walthamstow major
    129  (1.8%)  direction area

--- Most common 3-word phrases in 'article-4-di

In [ ]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

104 rows (1.4%) look like bare reference codes in 'article-4-direction-area'


,entity,organisation-entity,name
139,7010002142,67,59
140,7010002143,67,60
141,7010002144,67,61
142,7010002145,67,62
143,7010002146,67,63
...,...,...,...
7270,7010010438,153,0164B1
7271,7010010439,153,0164B2
7272,7010010440,153,0164B3
7274,7010010442,153,2967


In [112]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

6 rows (0.1%) have a blank/missing name in 'article-4-direction-area'


,entity,organisation-entity,name
241,7010002397,67,NaN
283,7010002439,67,NaN
315,7010002471,67,NaN
333,7010002489,67,NaN
369,7010002525,67,NaN
7127,7010010295,72,NaN


In [115]:
# Swap in whatever phrase looked suspicious above
phrase = "Town and Country"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

57 rows in 'article-4-direction-area' contain 'Town and Country'


,entity,organisation-entity,name
341,7010002497,67,Direction made by Bucks County Council dated 2...
482,7010002640,111,"Article 4(1) Direction 2, 2007, Town and Count..."
483,7010002641,111,"Town and Country Planning Direction No. 6, 1975"
484,7010002642,111,"Article 4 Direction 3, 1990, Town and Country ..."
485,7010002643,111,Town and Country Planning Direction No. 1 1976...
486,7010002644,111,Town and Country Planning Direction No. 2 1980...
487,7010002645,111,Town and Country Planning Direction No. 1 1980...
488,7010002646,111,"Town and Country Planning Direction No. 2, 1978"
489,7010002647,111,"Article 4 Direction No. 1, 1992, Town and Coun..."
490,7010002648,111,Article 4(1) Direction of the Town and Country...


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "Modified Direction" #Modified Direction 1 - CAZ, Modified Direction 2 - KIBAs and WNCBC, Modified Direction 3 - Town Centres

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

214 rows in 'article-4-direction-area' contain 'Modified Direction'


,entity,organisation-entity,name
1065,7010003264,192,Modified Direction 1 - CAZ
1066,7010003265,192,Modified Direction 1 - CAZ
1067,7010003266,192,Modified Direction 1 - CAZ
1068,7010003267,192,Modified Direction 1 - CAZ
1069,7010003268,192,Modified Direction 1 - CAZ
1070,7010003269,192,Modified Direction 1 - CAZ
1071,7010003270,192,Modified Direction 1 - CAZ
1072,7010003271,192,Modified Direction 1 - CAZ
1073,7010003272,192,Modified Direction 1 - CAZ
1074,7010003273,192,Modified Direction 1 - CAZ


In [133]:
# Swap in whatever phrase looked suspicious above
phrase = "Borough Employment Area"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

54 rows in 'article-4-direction-area' contain 'Borough Employment Area'


,entity,organisation-entity,name
2837,7010005998,366,Borough Employment Area related A4D
2838,7010005999,366,Borough Employment Area related A4D
2839,7010006000,366,Borough Employment Area related A4D
2840,7010006001,366,Borough Employment Area related A4D
2841,7010006002,366,Borough Employment Area related A4D
2842,7010006003,366,Borough Employment Area related A4D
2844,7010006005,366,Borough Employment Area related A4D
2845,7010006006,366,Borough Employment Area related A4D
2846,7010006007,366,Borough Employment Area related A4D
2847,7010006008,366,Borough Employment Area related A4D


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "Article 4 - No"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

84 rows in 'article-4-direction-area' contain 'Article 4 - No'


,entity,organisation-entity,name
2436,7010004790,182,Article 4 - No 3
2437,7010004791,182,Article 4 - No 5
2438,7010004792,182,Article 4 - No 8
2439,7010004793,182,Article 4 - No 6
2440,7010004794,182,Article 4 - No 7
2441,7010004795,182,Article 4 - No 9
2442,7010004796,182,Article 4 - No 15
2443,7010004797,182,Article 4 - No 22
2444,7010004798,182,Article 4 - No 23
2445,7010004799,182,Article 4 - No 24


In [125]:
# Swap in whatever phrase looked suspicious above
phrase = "Article 4 Direction"

matches = df.loc[df['name'] == phrase]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

39 rows in 'article-4-direction-area' contain 'Article 4 Direction'


,entity,organisation-entity,name
1812,7010004012,171,Article 4 Direction
1813,7010004013,171,Article 4 Direction
1814,7010004014,171,Article 4 Direction
1815,7010004015,171,Article 4 Direction
1816,7010004016,171,Article 4 Direction
1817,7010004017,171,Article 4 Direction
1818,7010004018,171,Article 4 Direction
1819,7010004019,171,Article 4 Direction
1820,7010004020,171,Article 4 Direction
1822,7010004022,171,Article 4 Direction


#### Conservation area

In [101]:
dataset = "conservation-area" # article-4-direction-area, conservation-area, listed-building-outlines, tree-preservation-zone, trees

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'conservation-area' ---
   1526  (14.0%)  area
   1483  (13.6%)  conservation
    451  (4.1%)  road
    402  (3.7%)  and
    337  (3.1%)  st
    330  (3.0%)  street
    300  (2.8%)  park
    248  (2.3%)  the
    242  (2.2%)  green
    206  (1.9%)  town
    194  (1.8%)  village
    191  (1.8%)  hill
    164  (1.5%)  west
    147  (1.3%)  centre
    145  (1.3%)  church

--- Most common 2-word phrases in 'conservation-area' ---
   1477  (13.6%)  conservation area
    115  (1.1%)  town centre
     57  (0.5%)  on the
     55  (0.5%)  high street
     40  (0.4%)  road conservation
     40  (0.4%)  area 1
     37  (0.3%)  area 2
     33  (0.3%)  the hill
     29  (0.3%)  road and
     27  (0.2%)  area tab
     27  (0.2%)  st albans
     25  (0.2%)  village conservation
     23  (0.2%)  area 3
     22  (0.2%)  street conservation
     22  (0.2%)  green conservation

--- Most common 3-word phrases in 'conservation-area' ---
     40  (0.4%)  road conservation ar

In [102]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

10 rows (0.1%) look like bare reference codes in 'conservation-area'


,entity,organisation-entity,name
5058,44008744,1,81
5062,44008749,169,3B
5064,44008755,169,63
5077,44008775,175,20
5079,44008781,175,25
5080,44008782,175,26
5081,44008783,175,1
5084,44008797,175,17
5118,44008849,198,0
5259,44009076,169,49


In [103]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

110 rows (1.0%) have a blank/missing name in 'conservation-area'


,entity,organisation-entity,name
5053,44008712,181,NaN
5055,44008729,65,NaN
5056,44008741,65,NaN
5057,44008743,65,NaN
5059,44008745,65,NaN
...,...,...,...
5239,44009054,150,NaN
5240,44009055,150,NaN
5241,44009056,150,NaN
5243,44009058,329,NaN


In [109]:
# Swap in whatever phrase looked suspicious above
phrase = "area"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


1530 rows in 'conservation-area' contain 'area'


,entity,organisation-entity,name
34,44000035,260,Ogden Conservation Area
35,44000036,260,Wardle Conservation Area
36,44000037,260,Ashworth Fold Conservation Area
37,44000038,260,Clegg Village Conservation Area
38,44000039,260,Rakewood Conservation Area
39,44000040,260,Hollingworth Fold Conservation Area
40,44000041,260,Whittaker Conservation Area
41,44000042,260,Butterworth Hall (Municipal Buildings) Conserv...
42,44000043,260,"Butterworth Hall, Newhey, Conservation Area"
43,44000044,260,Dearnley Workhouse Conservation Area


#### Listed building outline

In [13]:
dataset = "listed-building-outline" # article-4-direction-area, conservation-area, listed-building-outlines, tree-preservation-zone, trees

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'listed-building-outline' ---
  20306  (16.4%)  of
  20091  (16.2%)  street
  18873  (15.2%)  and
  16126  (13.0%)  house
  16106  (13.0%)  road
  15490  (12.5%)  the
  11464  (9.2%)  church
  10020  (8.1%)  st
   9903  (8.0%)  to
   9724  (7.8%)  farmhouse
   9435  (7.6%)  cottage
   8226  (6.6%)  south
   7382  (6.0%)  west
   7320  (5.9%)  north
   6767  (5.5%)  east

--- Most common 2-word phrases in 'listed-building-outline' ---
   5881  (4.7%)  church of
   5321  (4.3%)  of st
   4872  (3.9%)  high street
   3485  (2.8%)  east of
   3361  (2.7%)  west of
   2276  (1.8%)  south of
   2192  (1.8%)  and attached
   2030  (1.6%)  the old
   1844  (1.5%)  north of
   1679  (1.4%)  cheltenham gloucestershire
   1616  (1.3%)  saffron walden
   1610  (1.3%)  st mary
   1587  (1.3%)  street london
   1573  (1.3%)  of the
   1514  (1.2%)  walden essex

--- Most common 3-word phrases in 'listed-building-outline' ---
   4766  (3.8%)  church of st
   1514  (1

In [14]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

144 rows (0.1%) look like bare reference codes in 'listed-building-outline'


,entity,organisation-entity,name
1710,42112191,75,11/802
4379,42115994,90,No.8
4888,42116503,90,56
6088,42117703,90,51
7151,42118766,111,77
...,...,...,...
92202,42208065,112,31
92951,42208814,112,605
92952,42208815,112,603
92953,42208816,112,599


In [96]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

2245 rows (1.8%) have a blank/missing name in 'listed-building-outline'


,entity,organisation-entity,name
10380,42122307,358,NaN
10381,42122308,358,NaN
10382,42122309,358,NaN
10383,42122310,358,NaN
10384,42122311,358,NaN
...,...,...,...
119919,42235783,224,NaN
119920,42235784,224,NaN
119921,42235785,224,NaN
119922,42235786,224,NaN


In [97]:
# Swap in whatever phrase looked suspicious above
phrase = "No given name"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


220 rows in 'listed-building-outline' contain 'No given name'


,entity,organisation-entity,name
3366,42114980,329,No given name
3368,42114982,329,No given name
3372,42114986,329,No given name
3373,42114987,329,No given name
3374,42114988,329,No given name
3379,42114993,329,No given name
3380,42114994,329,No given name
3381,42114995,329,No given name
3385,42114999,329,No given name
3387,42115001,329,No given name


In [98]:
# Swap in whatever phrase looked suspicious above
phrase = "No name for this Entry"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


845 rows in 'listed-building-outline' contain 'No name for this Entry'


,entity,organisation-entity,name
61289,42177151,123,No name for this Entry
61303,42177165,123,No name for this Entry
61306,42177168,123,No name for this Entry
61324,42177186,123,No name for this Entry
61354,42177216,123,No name for this Entry
61357,42177219,123,No name for this Entry
61358,42177220,123,No name for this Entry
61384,42177246,123,No name for this Entry
61450,42177312,123,No name for this Entry
61457,42177319,123,No name for this Entry


In [100]:
# Swap in whatever phrase looked suspicious above
phrase = "No address supplied"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


149 rows in 'listed-building-outline' contain 'No address supplied'


,entity,organisation-entity,name
93453,42209316,192,No Address Supplied
93483,42209346,192,No Address Supplied
93484,42209347,192,No Address Supplied
93505,42209368,192,No Address Supplied
93516,42209379,192,No Address Supplied
93517,42209380,192,No Address Supplied
93523,42209386,192,No Address Supplied
93532,42209395,192,No Address Supplied
93553,42209416,192,No Address Supplied
93554,42209417,192,No Address Supplied


#### Tree Preservation Zone

In [66]:
dataset = "tree-preservation-zone" # article-4-direction-area, conservation-area, listed-building-outlines, tree-preservation-zone, trees

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'tree-preservation-zone' ---
  21903  (28.1%)  the
  19053  (24.5%)  tree
  18839  (24.2%)  preservation
  18782  (24.1%)  order
  14016  (18.0%)  no
  13109  (16.8%)  road
  12551  (16.1%)  council
  11868  (15.2%)  of
  10680  (13.7%)  district
   7394  (9.5%)  borough
   7390  (9.5%)  and
   6416  (8.2%)  tpo
   6159  (7.9%)  land
   4995  (6.4%)  1
   4930  (6.3%)  broadland

--- Most common 2-word phrases in 'tree-preservation-zone' ---
  18762  (24.1%)  tree preservation
  17721  (22.8%)  preservation order
   8978  (11.5%)  district council
   6692  (8.6%)  council tree
   4925  (6.3%)  broadland district
   4845  (6.2%)  the broadland
   4613  (5.9%)  borough of
   3552  (4.6%)  made under
   3333  (4.3%)  order no
   3269  (4.2%)  town and
   3203  (4.1%)  country planning
   3191  (4.1%)  and country
   3106  (4.0%)  the town
   3027  (3.9%)  under the
   2773  (3.6%)  borough council

--- Most common 3-word phrases in 'tree-preservation-zone

In [67]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

12050 rows (11.8%) look like bare reference codes in 'tree-preservation-zone'


,entity,organisation-entity,name
9444,19113163,152,W1
9445,19113164,152,W1
9446,19113165,152,W1
9447,19113166,152,W2
9448,19113167,152,W1
...,...,...,...
66425,19178244,205,5001/2025/TPO
66426,19178245,205,5002/2025/TPO
66427,19178246,205,5004/2025/TPO
66428,19178247,205,5006/2025/TPO


In [ ]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

In [69]:
# Swap in whatever phrase looked suspicious above
phrase = "tree preservation order no"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


1918 rows in 'tree-preservation-zone' contain 'tree preservation order no'


,entity,organisation-entity,name
11771,19116640,48,The Urban District Council of Potters Bar Land...
12099,19116968,48,Tree Preservation Order No. 1 of 1953 made und...
12100,19116969,48,Tree Preservation Order No. 1 of 1953 made und...
12101,19116970,48,Tree Preservation Order No. 1 of 1953 made und...
12513,19117382,48,County of Middlesex Urban District of Friern B...
12514,19117383,48,"County of Middlesex, Urban District of Friern ..."
22002,19132718,67,The Bucks County Council (Amersham Rural Distr...
22003,19132719,67,The Bucks County Council (Amersham Rural Distr...
22004,19132720,67,The Bucks County Council (Amersham Rural Distr...
22005,19132721,67,The Bucks County Council (Amersham Rural Distr...


#### Tree

In [5]:
dataset = "tree" # article-4-direction-area, conservation-area, listed-building-outlines, tree-preservation-zone, trees

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'tree' ---
  44657  (21.1%)  tree
  41159  (19.4%)  preservation
  38317  (18.1%)  road
  35059  (16.5%)  the
  33800  (15.9%)  order
  30827  (14.5%)  of
  26543  (12.5%)  no
  19701  (9.3%)  tpo
  18425  (8.7%)  land
  17160  (8.1%)  borough
  16907  (8.0%)  council
  16907  (8.0%)  and
  13633  (6.4%)  lane
  12538  (5.9%)  district
  12261  (5.8%)  west

--- Most common 2-word phrases in 'tree' ---
  41153  (19.4%)  tree preservation
  31498  (14.8%)  preservation order
  11812  (5.6%)  borough of
   9894  (4.7%)  order no
   9069  (4.3%)  district council
   8689  (4.1%)  land at
   7893  (3.7%)  west sussex
   7779  (3.7%)  made under
   7297  (3.4%)  preservation orders
   6572  (3.1%)  201 21
   6062  (2.9%)  country planning
   6034  (2.8%)  the town
   6032  (2.8%)  town and
   5999  (2.8%)  under the
   5950  (2.8%)  and country

--- Most common 3-word phrases in 'tree' ---
  31492  (14.8%)  tree preservation order
   9829  (4.6%)  preservat

In [ ]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

43535 rows (17.0%) look like bare reference codes in 'tree'


,entity,organisation-entity,name
9103,7002009103,228,T1
9104,7002009104,228,T1
9105,7002009105,228,T1
9106,7002009106,228,T1
9107,7002009107,228,T2
...,...,...,...
204841,7002211483,26,13.53/1/12/L
204842,7002211484,26,13.53/1/12/L
205094,7002211736,26,13.53/2/92/SU
235005,7002241650,351,08/13


In [ ]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]


In [64]:
# Swap in whatever phrase looked suspicious above
phrase = "tree preservation order no"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


6771 rows in 'tree' contain 'tree preservation order no'


,entity,organisation-entity,name
72,7002000072,67,Tree Preservation Order No 11 of 2013
76,7002000076,67,Tree Preservation Order No 11 of 2013
95,7002000095,67,Tree Preservation Order No 10 of 2014
96,7002000096,67,Tree Preservation Order No 11 of 2014
97,7002000097,67,Tree Preservation Order No 11 of 2014
98,7002000098,67,Tree Preservation Order No 11 of 2014
99,7002000099,67,Tree Preservation Order No 11 of 2014
100,7002000100,67,Tree Preservation Order No 11 of 2014
101,7002000101,67,Tree Preservation Order No 11 of 2014
102,7002000102,67,Tree Preservation Order No 11 of 2014


In [65]:
# Swap in whatever phrase looked suspicious above
phrase = "tpo"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


20089 rows in 'tree' contain 'tpo'


,entity,organisation-entity,name
27,7002000027,67,TPO No 8 of 2011
47,7002000047,67,TPO 4 of 2012
48,7002000048,67,TPO 4 of 2012
49,7002000049,67,TPO 4 of 2012
50,7002000050,67,TPO 4 of 2012
51,7002000051,67,TPO 4 of 2012
52,7002000052,67,TPO 4 of 2012
53,7002000053,67,TPO No 6 of 2012
54,7002000054,67,TPO No 6 of 2012
55,7002000055,67,TPO No 3 of 2013


In [8]:
# Swap in whatever phrase looked suspicious above
phrase = "Tree Preservation Order 1995"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


684 rows in 'tree' contain 'Tree Preservation Order 1995'


,entity,organisation-entity,name
658,7002000658,67,The Chiltern District Council (Land at Traffor...
659,7002000659,67,The Chiltern District Council (Land at Traffor...
660,7002000660,67,The Chiltern District Council (Land at Traffor...
661,7002000661,67,The Chiltern District Council (Land at Traffor...
662,7002000662,67,The Chiltern District Council (Land at Traffor...
663,7002000663,67,The Chiltern District Council (Land at Traffor...
664,7002000664,67,The Chiltern District Council (Land at Traffor...
665,7002000665,67,The Chiltern District Council (Land at Traffor...
666,7002000666,67,The Chiltern District Council (Land at Traffor...
667,7002000667,67,The Chiltern District Council (Land at Traffor...


In [12]:
# Swap in whatever phrase looked suspicious above
phrase = "Town and Country"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

5949 rows in 'tree' contain 'Town and Country'


,entity,organisation-entity,name
2196,7002002196,67,The Eton Rural (Burnham Beeches) Town and Coun...
6225,7002006225,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6226,7002006226,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6227,7002006227,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6228,7002006228,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6229,7002006229,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6230,7002006230,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6231,7002006231,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6232,7002006232,67,Eton R.D.C. Tree Preservation Order (No.1) dat...
6233,7002006233,67,Eton R.D.C. Tree Preservation Order (No.1) dat...


In [7]:
dupes = duplicate_names(df)
dupes.head(20)

name
Tree Preservation Orders                                                                                4794
T1                                                                                                      3307
https://services.southwark.gov.uk/environment/trees/tree-preservation-orders-and-conservation-areas     2503
Protected Individual Tree                                                                               2154
T2                                                                                                      2014
T3                                                                                                      1474
T4                                                                                                      1179
T5                                                                                                       962
T6                                                                                                       818
T7            